# 6.4 v2 — Otimização Econômica: versão interpretável

Este notebook replica a análise de `6.4_otimizacao_economica.ipynb`, mas expõe
cada passo do raciocínio de forma explícita.

**Pergunta central:** dado o preço do metanol ($P_M$) e o custo da energia ($P_E$),
qual dos 23 pontos Pareto-ótimos maximiza o lucro horário?

---

## Por que isso é um problema de otimização sobre a fronteira de Pareto?

A Etapa 6.3 gerou uma fronteira de Pareto: o conjunto de soluções onde **não é
possível reduzir ET sem sacrificar M** (e vice-versa). Cada ponto dessa fronteira
é tecnicamente eficiente — nenhum é *melhor* que outro sem um critério econômico.

A função de lucro é esse critério: ela converte o trade-off físico (kW vs kg/hr)
em uma escala única (USD/hr), permitindo comparar os pontos e escolher um.

$$
\text{lucro}(i) = M_{CH_3OH}(i) \times P_M - ET(i) \times P_E \quad [\text{USD/hr}]
$$

O ponto ótimo é simplesmente `argmax(lucro)` sobre os 23 pontos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

## Seção 1 — Fronteira de Pareto

Carregamos os 23 pontos gerados em 6.3. Cada linha é uma solução Pareto-ótima:
- **ET** (kW): consumo de energia — queremos minimizar
- **M_CH3OH** (kg/hr): produção de metanol — queremos maximizar
- **x_CH3OH**: pureza (≥ 0.98 em todos os pontos — restrição ativa)

O CSV está ordenado por ET crescente: **índice 0 = mínimo ET, índice 22 = máximo M**.

In [ ]:
df = pd.read_csv('../../ETAPA_6/6.3/6.3_pareto_solucoes.csv')

assert df.isnull().sum().sum() == 0
assert (df['x_CH3OH'] >= 0.98).all()
assert df['ET'].is_monotonic_increasing

print(f'Pontos na fronteira: {len(df)}')
print(f'ET:      {df["ET"].min():.0f} – {df["ET"].max():.0f} kW')
print(f'M_CH3OH: {df["M_CH3OH"].min():.0f} – {df["M_CH3OH"].max():.0f} kg/hr')
print()

# Mostrar apenas as colunas de interesse, com índice explícito
df[['ET', 'M_CH3OH', 'x_CH3OH']].round(1)

### Visualização da fronteira

A fronteira tem forma côncava: ao aumentar ET, M cresce, mas com retorno
decrescente — cada kW extra de energia compra cada vez menos kg/hr de metanol.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df['ET'], df['M_CH3OH'], '-o', color='#1f77b4', lw=2, ms=5, zorder=3)

# Anotar cada ponto com seu índice
for i, row in df.iterrows():
    ax.annotate(str(i), (row['ET'], row['M_CH3OH']),
                textcoords='offset points', xytext=(4, 4), fontsize=7, color='#333')

ax.set_xlabel('Consumo de energia ET (kW)', fontsize=11)
ax.set_ylabel('Produção de metanol $M_{CH_3OH}$ (kg/hr)', fontsize=11)
ax.set_title('Fronteira de Pareto — 23 pontos (índice 0 = mín ET, índice 22 = máx M)', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Seção 2 — Calculando o lucro para cada ponto

### Parâmetros econômicos

Vamos trabalhar com o **cenário médio** primeiro:
- $P_M = 0.40$ USD/kg (preço de venda do metanol)
- $P_E = 0.10$ USD/kWh (custo da energia elétrica)
- $\alpha = P_E / P_M = 0.25$ kg/kWh (razão de preços — a variável central da análise)

O lucro de cada ponto é calculado independentemente:

$$
L_i = M_i \times 0.40 - ET_i \times 0.10
$$

Abaixo mostramos o cálculo explícito para todos os 23 pontos.

In [ ]:
P_M = 0.40   # USD/kg
P_E_medio = 0.10  # USD/kWh
alpha_medio = P_E_medio / P_M  # 0.25

print(f'P_M = {P_M} USD/kg')
print(f'P_E = {P_E_medio} USD/kWh')
print(f'α   = P_E / P_M = {alpha_medio:.3f} kg/kWh')
print()

# Calcular cada componente do lucro separadamente
analise = df[['ET', 'M_CH3OH']].copy()
analise['receita (M × P_M)']  = (analise['M_CH3OH'] * P_M).round(1)
analise['custo_E (ET × P_E)'] = (analise['ET'] * P_E_medio).round(1)
analise['lucro (USD/hr)']     = (analise['receita (M × P_M)'] - analise['custo_E (ET × P_E)']).round(1)

# Identificar o ótimo
idx_otimo = analise['lucro (USD/hr)'].idxmax()

print(f'Ponto com maior lucro: índice {idx_otimo}')
print(f'Lucro máximo: {analise.loc[idx_otimo, "lucro (USD/hr)"]:.1f} USD/hr')
print()

# Destacar o ótimo na tabela
print('Tabela completa de lucros (↑ = ótimo):')
display_df = analise.copy()
display_df.index.name = 'ponto'

def highlight_max(s):
    return ['background-color: #d4edda' if v == s.max() else '' for v in s]

display_df.style.apply(highlight_max, subset=['lucro (USD/hr)'])

### Por que o ponto 8 vence e não o 15 (que tem mais M)?

O ponto 15 tem mais metanol, mas consome muito mais energia.
A célula abaixo compara os dois explicitamente.

In [ ]:
p8  = df.iloc[8]
p15 = df.iloc[15]

delta_M  = p15['M_CH3OH'] - p8['M_CH3OH']
delta_ET = p15['ET']      - p8['ET']

receita_extra = delta_M  * P_M
custo_extra   = delta_ET * P_E_medio
lucro_marginal = receita_extra - custo_extra

print('Comparação ponto 8 vs ponto 15:')
print(f'  ΔM_CH3OH = {delta_M:+.1f} kg/hr  →  receita extra = {receita_extra:+.1f} USD/hr')
print(f'  ΔET      = {delta_ET:+.1f} kW    →  custo extra   = {custo_extra:+.1f} USD/hr')
print(f'  Lucro marginal de ir do ponto 8 ao 15: {lucro_marginal:+.1f} USD/hr')
print()
if lucro_marginal < 0:
    print('→ Ir para o ponto 15 REDUZ o lucro: o custo energético extra supera a receita extra.')
else:
    print('→ Ir para o ponto 15 AUMENTA o lucro.')

## Seção 3 — Geometria da iso-lucro

### De onde vem a "curva" de iso-lucro?

Fixe o lucro em um valor $L^*$ e isole $M$ na equação de lucro:

$$
L^* = M \times P_M - ET \times P_E
\quad\Rightarrow\quad
M = \underbrace{\alpha}_{\text{inclinação}} \times ET + \underbrace{\frac{L^*}{P_M}}_{\text{intercepto}}
$$

Isso é a equação de uma **reta** no plano (ET, M), com:
- **Inclinação**: $\alpha = P_E / P_M$ — determinada pela razão de preços
- **Intercepto**: $L^* / P_M$ — determinado pelo nível de lucro

Retas paralelas (mesma inclinação $\alpha$, interceptos diferentes) representam
**níveis diferentes de lucro**. Retas mais acima = lucro maior.

### Como encontrar o ótimo geometricamente?

Imagine deslocar a reta de iso-lucro para cima (aumentar o intercepto = aumentar $L^*$).
O **último ponto da fronteira tocado** antes de a reta "sair" da fronteira é o ótimo.
Matematicamente: é onde a iso-lucro tangencia a fronteira côncava.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Fronteira
ax.plot(df['ET'], df['M_CH3OH'], '-o', color='#1f77b4', lw=2, ms=5, zorder=3,
        label='Fronteira de Pareto')

# Anotar índices dos pontos
for i, row in df.iterrows():
    ax.annotate(str(i), (row['ET'], row['M_CH3OH']),
                textcoords='offset points', xytext=(4, 3), fontsize=7, color='#333')

# Iso-lucros para α médio: L* = lucro do ótimo + offset
p_otimo = df.iloc[idx_otimo]
L_otimo = p_otimo['M_CH3OH'] * P_M - p_otimo['ET'] * P_E_medio

et_vec = np.linspace(df['ET'].min() * 0.5, df['ET'].max() * 1.05, 400)
m_span = df['M_CH3OH'].max() - df['M_CH3OH'].min()

offsets = [0.35 * m_span, 0.0, -0.35 * m_span]
descricoes = ['lucro maior (reta acima — fora da fronteira)', 'reta ótima (tangente à fronteira)', 'lucro menor (reta abaixo)']
cores_iso = ['#2ca02c', 'orange', '#d62728']

for offset, desc, cor in zip(offsets, descricoes, cores_iso):
    L_star = L_otimo + offset
    intercepto = L_star / P_M
    M_iso = alpha_medio * et_vec + intercepto
    mask = (M_iso >= df['M_CH3OH'].min() * 0.65) & (M_iso <= df['M_CH3OH'].max() * 1.25)
    ax.plot(et_vec[mask], M_iso[mask], '--', lw=1.4, alpha=0.8, color=cor, label=f'Iso-lucro: {desc}')

# Ponto ótimo
ax.scatter(p_otimo['ET'], p_otimo['M_CH3OH'], s=250, marker='*', color='orange',
           zorder=5, edgecolors='k', linewidths=0.7,
           label=f'Ótimo α médio: ponto {idx_otimo}')

# Anotação da equação
ax.text(0.02, 0.97,
        f'Iso-lucro (α={alpha_medio:.2f}): M = {alpha_medio:.2f} × ET + L*/P_M',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('Consumo de energia ET (kW)', fontsize=11)
ax.set_ylabel('Produção de metanol $M_{CH_3OH}$ (kg/hr)', fontsize=11)
ax.set_title(f'Geometria da iso-lucro (α = {alpha_medio:.2f}, cenário médio)\n'
             'A reta ótima é a mais alta que ainda toca a fronteira', fontsize=10)
ax.legend(fontsize=8.5, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Inclinação da iso-lucro: α = {alpha_medio:.3f} (adimensional em P_M=1, ou kg/kWh)')
print(f'Intercepto da iso-lucro ótima: L*/P_M = {L_otimo/P_M:.1f} kg/hr')

## Seção 4 — Como α define o ponto ótimo

### A lógica da TMS (Taxa Marginal de Substituição)

Entre dois pontos consecutivos da fronteira, a TMS responde:

> *"Para reduzir ET em 1 kW, quanto M se perde?"*

$$
\text{TMS}_{i \to i+1} = \frac{\Delta M}{\Delta ET} = \frac{M_{i+1} - M_i}{ET_{i+1} - ET_i} \quad [\text{kg/kWh}]
$$

**Regra de decisão:** ao mover do ponto $i$ para $i+1$ (mais energia, mais metanol),
o lucro aumenta se e somente se:

$$
\text{TMS}_{i \to i+1} > \alpha
$$

Interpretação: "o ganho marginal de metanol supera o custo marginal da energia".

Quando $\alpha$ cresce (energia fica mais cara), mais trechos da fronteira passam a
ter TMS < α, e o ótimo recua para a esquerda (menos ET).

In [ ]:
M   = df['M_CH3OH'].values
ET  = df['ET'].values

# TMS entre cada par de pontos consecutivos
delta_M  = np.diff(M)
delta_ET = np.diff(ET)
TMS = delta_M / delta_ET  # kg/kWh

df_tms = pd.DataFrame({
    'trecho': [f'{i}→{i+1}' for i in range(len(df)-1)],
    'ΔET (kW)':  delta_ET.round(0),
    'ΔM (kg/hr)': delta_M.round(1),
    'TMS (kg/kWh)': TMS.round(4),
})

print('TMS entre pontos consecutivos da fronteira:')
print(df_tms.to_string(index=False))
print()
print(f'TMS mín: {TMS.min():.4f}  TMS máx: {TMS.max():.4f}')
print()
print('Interpretação dos limiares de α:')
print(f'  α < {TMS.min():.4f}  → ótimo sempre no ponto 22 (máx M) — energia quase gratuita')
print(f'  α > {TMS.max():.4f}  → ótimo sempre no ponto 0  (mín ET) — energia caríssima')
print(f'  α ∈ [{TMS.min():.4f}, {TMS.max():.4f}] → ótimo em ponto interior (caso dos 3 cenários reais)')

### Para cada α, qual ponto vence?

A célula abaixo mostra, passo a passo, como o algoritmo percorre a tabela de lucros
para três valores de α.

In [ ]:
P_E_cenarios = {'baixo': 0.05, 'medio': 0.10, 'alto': 0.15}

for nome, P_E in P_E_cenarios.items():
    alpha = P_E / P_M
    L = M * P_M - ET * P_E
    idx = int(np.argmax(L))

    print('=' * 60)
    print(f'Cenário: {nome}  |  P_E = {P_E} USD/kWh  |  α = {alpha:.3f} kg/kWh')
    print(f'Fórmula: L = M × {P_M} - ET × {P_E}')
    print()

    # Mostrar os 5 pontos com maior lucro
    ranking = pd.DataFrame({'ponto': range(len(L)), 'ET': ET.round(0), 'M': M.round(1), 'L (USD/hr)': L.round(1)})
    ranking = ranking.sort_values('L (USD/hr)', ascending=False)
    print('Top 5 pontos por lucro:')
    print(ranking.head(5).to_string(index=False))
    print()
    print(f'→ Ponto ótimo: {idx}  |  ET = {ET[idx]:.0f} kW  |  M = {M[idx]:.1f} kg/hr  |  lucro = {L[idx]:.1f} USD/hr')
    print()

    # Verificar via TMS: quais trechos são rentáveis?
    trechos_rentaveis = np.where(TMS > alpha)[0]
    if len(trechos_rentaveis) > 0:
        ultimo_rentavel = trechos_rentaveis[-1]
        print(f'  Via TMS: o último trecho rentável é {ultimo_rentavel}→{ultimo_rentavel+1} '
              f'(TMS={TMS[ultimo_rentavel]:.4f} > α={alpha:.3f})')
        print(f'  Logo o ótimo é o ponto {ultimo_rentavel+1} (ponto APÓS o último trecho rentável)')
    else:
        print(f'  Via TMS: nenhum trecho é rentável (todos TMS < α={alpha:.3f}) → ótimo no ponto 0')
    print()

### Visualizando como α muda a inclinação da iso-lucro

Os três cenários usam inclinações diferentes para a reta de iso-lucro.
Uma inclinação maior (α alto) significa que a reta "sobe mais rápido" à medida que
ET aumenta — então ela sai da fronteira mais cedo, com um ponto de tangência
mais à esquerda (menor ET).

In [ ]:
CORES = {'baixo': '#2ca02c', 'medio': '#ff7f0e', 'alto': '#d62728'}

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df['ET'], df['M_CH3OH'], '-o', color='#1f77b4', lw=2, ms=5, zorder=3,
        label='Fronteira de Pareto')

for i, row in df.iterrows():
    ax.annotate(str(i), (row['ET'], row['M_CH3OH']),
                textcoords='offset points', xytext=(4, 3), fontsize=7, color='#555')

et_vec = np.linspace(df['ET'].min() * 0.4, df['ET'].max() * 1.05, 400)

for nome, P_E in P_E_cenarios.items():
    alpha = P_E / P_M
    L = M * P_M - ET * P_E
    idx = int(np.argmax(L))
    L_otimo = L[idx]

    # Iso-lucro tangente
    M_iso = alpha * et_vec + L_otimo / P_M
    mask = (M_iso >= df['M_CH3OH'].min() * 0.6) & (M_iso <= df['M_CH3OH'].max() * 1.20)
    ax.plot(et_vec[mask], M_iso[mask], '--', lw=1.5, color=CORES[nome], alpha=0.85,
            label=f'Iso-lucro α={alpha:.3f} ({nome}), inclinação={alpha:.3f}')

    ax.scatter(ET[idx], M[idx], s=220, marker='*', color=CORES[nome],
               zorder=5, edgecolors='k', linewidths=0.6)
    ax.annotate(f'ótimo\nα={alpha:.3f}\nponto {idx}',
                (ET[idx], M[idx]), textcoords='offset points',
                xytext=(-60, -35), fontsize=8, color=CORES[nome],
                arrowprops=dict(arrowstyle='->', color=CORES[nome], lw=1))

ax.set_xlabel('Consumo de energia ET (kW)', fontsize=11)
ax.set_ylabel('Produção de metanol $M_{CH_3OH}$ (kg/hr)', fontsize=11)
ax.set_title('Iso-lucros para os 3 cenários: inclinação α diferente → ponto ótimo diferente', fontsize=10)
ax.legend(fontsize=8.5, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('6.4_pareto_economico_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('6.4_pareto_economico_v2.png salvo.')

## Seção 5 — Varredura contínua de α: mapa de deslocamento

Variando α de 0.05 a 0.60, identificamos qual ponto da fronteira é ótimo em cada valor.

**Por que o gráfico tem degraus e não é suave?**  
O ótimo só pode ser um dos 23 pontos discretos. Cada "degrau" é uma transição:
para α imediatamente acima de um valor crítico, o ponto ótimo pula de $i+1$ para $i$.
O valor crítico é exatamente a TMS do trecho $i \to i+1$.

In [ ]:
alpha_range = np.linspace(0.05, 0.60, 2000)
idx_por_alpha = []

for alpha in alpha_range:
    # Normalizar P_M=1 para isolar o efeito de α
    L = M * 1.0 - ET * alpha
    idx_por_alpha.append(int(np.argmax(L)))

idx_por_alpha = np.array(idx_por_alpha)

# Encontrar as transições: onde o ponto ótimo muda
transicoes = np.where(np.diff(idx_por_alpha) != 0)[0]
print('Transições de ótimo (α crítico → ponto ótimo muda de X para Y):')
for t in transicoes:
    alpha_crit = alpha_range[t]
    print(f'  α ≈ {alpha_crit:.4f}: ponto {idx_por_alpha[t]} → ponto {idx_por_alpha[t+1]}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.step(alpha_range, idx_por_alpha, color='#1f77b4', lw=2, where='post',
        label='Índice do ponto Pareto ótimo')

# Linhas dos 3 cenários
for nome, P_E in P_E_cenarios.items():
    alpha_c = P_E / P_M
    L_c = M * 1.0 - ET * alpha_c
    idx_c = int(np.argmax(L_c))
    ax.axvline(x=alpha_c, color=CORES[nome], lw=1.5, ls='--', alpha=0.85,
               label=f'α={alpha_c:.3f} ({nome}) → ponto {idx_c}')
    ax.scatter([alpha_c], [idx_c], color=CORES[nome], s=90, zorder=5,
               edgecolors='k', linewidths=0.5)

# Anotar os degraus com os valores de TMS
for t in transicoes:
    alpha_crit = alpha_range[t]
    y_pos = idx_por_alpha[t+1] + 0.5
    ax.axvline(x=alpha_crit, color='gray', lw=0.7, ls=':', alpha=0.5)

ax.set_xlabel('Razão de preços α = P_E / P_M (kg/kWh)', fontsize=11)
ax.set_ylabel('Índice do ponto ótimo\n(0 = mín ET  |  22 = máx M)', fontsize=10)
ax.set_title('Deslocamento do ótimo econômico em função de α\n'
             'Degraus = transições de ponto ótimo (ocorrem em α = TMS local)', fontsize=10)
ax.set_xlim(alpha_range[0], alpha_range[-1])
ax.set_yticks(range(0, len(df), 2))
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('6.4_mapa_alpha_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('6.4_mapa_alpha_v2.png salvo.')

### Confirmando: os degraus coincidem com a TMS local

Cada transição no mapa de α deveria ocorrer exatamente em α = TMS do trecho
correspondente. Verificamos isso abaixo.

In [ ]:
print('Verificação: α crítico vs TMS local\n')
print(f'{"Transição":<15} {"α crítico (mapa)":>20} {"TMS trecho":>15} {"Coincide?":>12}')
print('-' * 65)

for t in transicoes:
    de = idx_por_alpha[t]
    para = idx_por_alpha[t + 1]
    alpha_crit = alpha_range[t]
    # TMS do trecho que conecta os dois pontos
    trecho_idx = min(de, para)  # o trecho i→i+1 que separa os dois pontos
    if trecho_idx < len(TMS):
        tms_local = TMS[trecho_idx]
        coincide = abs(alpha_crit - tms_local) < 0.01
        print(f'{de}→{para:<12} {alpha_crit:>20.4f} {tms_local:>15.4f} {"✓" if coincide else "≈":>12}')

print()
print('Conclusão: cada degrau no mapa de α ocorre onde α = TMS do trecho correspondente.')

## Seção 6 — Tabela-resumo dos três cenários

In [ ]:
resultados = []
for nome, P_E in P_E_cenarios.items():
    alpha = P_E / P_M
    L = M * P_M - ET * P_E
    idx = int(np.argmax(L))
    row = df.iloc[idx].to_dict()
    row.update({
        'cenario': nome, 'alpha': alpha, 'P_E': P_E,
        'idx_pareto': idx, 'lucro_USD_hr': round(L[idx], 1)
    })
    resultados.append(row)

df_res = pd.DataFrame(resultados)
df_res[['cenario', 'alpha', 'P_E', 'idx_pareto', 'ET', 'M_CH3OH', 'x_CH3OH', 'lucro_USD_hr']].round({
    'alpha': 3, 'ET': 0, 'M_CH3OH': 1, 'x_CH3OH': 4, 'lucro_USD_hr': 1
})